# 🤖 AI Tools Usage Survey — GPA Analysis
**Course:** Data Science & Machine Learning  
**Target Variable:** GPA (Binary: ≥ 3.5 = High, < 3.5 = Low)  
**Model:** Logistic Regression (full dataset, no train/test split)

---
### 📋 Table of Contents
1. [Import Libraries](#1-import-libraries)
2. [Load & Inspect Data](#2-load--inspect-data)
3. [Data Cleaning](#3-data-cleaning)
4. [Exploratory Data Analysis (EDA)](#4-exploratory-data-analysis)
5. [Feature Engineering](#5-feature-engineering)
6. [Logistic Regression Model](#6-logistic-regression-model)
7. [Model Evaluation](#7-model-evaluation)
8. [Conclusion](#8-conclusion)


## 1. Import Libraries
> We import all required libraries upfront. `pandas` for data manipulation, `sklearn` for modeling, `matplotlib`/`seaborn` for visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    ConfusionMatrixDisplay
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
matplotlib.rcParams['figure.dpi'] = 120
print("✅ Libraries imported successfully")

## 2. Load & Inspect Data
> The CSV has 2 decorative header rows before the actual column names on row 3.  
> We use `skiprows=2` to skip them, then inspect shape, dtypes, and sample rows.


In [ ]:
# ── Load raw CSV ──────────────────────────────────────────────────────────
df_raw = pd.read_csv("AI_Survey_Dataset_tp3.csv", skiprows=2)

print("Shape:", df_raw.shape)
print("\nColumns:")
for c in df_raw.columns:
    print(" •", repr(c))

In [ ]:
# Preview first 5 rows
df_raw.head()

In [ ]:
# Data types & null counts
print("Dtypes & Non-Null Counts:")
df_raw.info()

## 3. Data Cleaning

### Steps:
| Step | Action | Reason |
|------|--------|--------|
| 1 | Rename columns | Original names are multi-line, unreadable |
| 2 | Drop `No.` column | Row index — not a feature |
| 3 | Drop fully-empty rows | Safety check |
| 4 | Drop rows with missing GPA | Target must not be null |
| 5 | Verify no remaining nulls | Confirm clean dataset |

> **Note:** Zero missing values were found after fixing the header rows. The dataset is complete (57 × 7).


In [ ]:
# ── Rename columns ────────────────────────────────────────────────────────
COLUMN_NAMES = {
    df_raw.columns[0]: "No",
    df_raw.columns[1]: "ai_frequency",
    df_raw.columns[2]: "hours_per_week",
    df_raw.columns[3]: "copy_behavior",
    df_raw.columns[4]: "improved_grades",
    df_raw.columns[5]: "cannot_complete_without_ai",
    df_raw.columns[6]: "critical_thinking",
    df_raw.columns[7]: "gpa",
}

df = df_raw.rename(columns=COLUMN_NAMES).copy()
print("Renamed columns:", df.columns.tolist())

In [ ]:
# ── Drop index column & empty rows ────────────────────────────────────────
df.drop(columns=["No"], inplace=True)
df.dropna(how="all", inplace=True)
df.dropna(subset=["gpa"], inplace=True)

print(f"Shape after cleaning: {df.shape}")
print("\nMissing values per column:")
print(df.isnull().sum())

In [ ]:
df.head(8)

In [ ]:
# Descriptive statistics for numeric columns
df.describe()

## 4. Exploratory Data Analysis

### What we'll explore:
- **GPA distribution** — understand the target variable
- **Categorical value counts** — see how students use AI
- **Average GPA by category** — visual relationship between features and GPA
- **Correlation heatmap** — after encoding, see linear relationships

> 💡 Key insight: AI frequency alone doesn't strongly predict GPA. The *perception* that AI improved grades (Likert scale) is the strongest single correlate with GPA.


In [ ]:
# ── Value counts for categorical columns ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Categorical Feature Distributions", fontsize=14, fontweight="bold")

cats = {
    "ai_frequency":  (["Never","Sometimes","Often","Daily"], axes[0]),
    "hours_per_week":(["0 hours","less than 1 hours","1-3 hours","4-6 hours","More than 6 hours"], axes[1]),
    "copy_behavior": (["Never","Rarely","Sometimes","Often","Always"], axes[2]),
}

LIME = "#C9F01E"; DARK = "#1A1A2E"
colors = [LIME, "#a8d400", "#82a800", "#5c7800", "#364800"]

for col, (order, ax) in cats.items():
    counts = df[col].value_counts().reindex(order, fill_value=0)
    ax.bar(counts.index, counts.values, color=colors[:len(order)], edgecolor=DARK, linewidth=1)
    ax.set_title(col.replace("_", " ").title(), fontweight="bold")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=25)
    for i, v in enumerate(counts.values):
        ax.text(i, v + 0.2, str(v), ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# ── GPA Distribution ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(df["gpa"], bins=8, color=LIME, edgecolor=DARK, linewidth=1.5)
axes[0].axvline(df["gpa"].mean(), color="red", linestyle="--", linewidth=2, label=f"Mean = {df['gpa'].mean():.2f}")
axes[0].set_title("GPA Distribution", fontweight="bold")
axes[0].set_xlabel("GPA")
axes[0].set_ylabel("Count")
axes[0].legend()

# Boxplot
axes[1].boxplot(df["gpa"], vert=True, patch_artist=True,
                boxprops=dict(facecolor=LIME, color=DARK),
                medianprops=dict(color="red", linewidth=2))
axes[1].set_title("GPA Boxplot", fontweight="bold")
axes[1].set_ylabel("GPA")
axes[1].set_xticks([1])
axes[1].set_xticklabels(["GPA"])

plt.suptitle(f"GPA Summary  |  Min={df['gpa'].min()}  Max={df['gpa'].max()}  Mean={df['gpa'].mean():.2f}  Std={df['gpa'].std():.2f}", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Avg GPA by Categorical Features ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Average GPA by Categorical Feature", fontsize=13, fontweight="bold")

orders = {
    "ai_frequency":  ["Never","Sometimes","Often","Daily"],
    "copy_behavior": ["Never","Rarely","Sometimes","Often","Always"],
    "hours_per_week":["0 hours","less than 1 hours","1-3 hours","4-6 hours","More than 6 hours"],
}
ax_list = axes.flatten()
bar_colors = ["#1A1A2E","#028090","#6D2E46"]

for idx, (col, order) in enumerate(orders.items()):
    means = df.groupby(col)["gpa"].mean().reindex(order).dropna()
    ax_list[idx].bar(means.index, means.values, color=bar_colors[idx], edgecolor="white", linewidth=1.2)
    ax_list[idx].set_title(col.replace("_"," ").title(), fontweight="bold")
    ax_list[idx].set_ylabel("Avg GPA")
    ax_list[idx].set_ylim(2.5, 4.0)
    ax_list[idx].tick_params(axis="x", rotation=25)
    for i, v in enumerate(means.values):
        ax_list[idx].text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9, fontweight="bold", color="white" if bar_colors[idx] == "#1A1A2E" else "black")

plt.tight_layout()
plt.show()

In [ ]:
# ── Likert Scale vs GPA ───────────────────────────────────────────────────
likert_cols = ["improved_grades", "cannot_complete_without_ai", "critical_thinking"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Avg GPA by Likert Scale Response (1–5)", fontsize=13, fontweight="bold")
pal = ["#C9F01E","#F96167","#028090"]

for i, col in enumerate(likert_cols):
    means = df.groupby(col)["gpa"].mean()
    axes[i].bar(means.index, means.values, color=pal[i], edgecolor=DARK, linewidth=1.2)
    axes[i].set_title(col.replace("_"," ").title(), fontweight="bold")
    axes[i].set_xlabel("Rating (1=Strongly Disagree, 5=Strongly Agree)")
    axes[i].set_ylabel("Avg GPA")
    axes[i].set_ylim(2.5, 4.0)
    axes[i].set_xticks([1,2,3,4,5])
    for x, v in zip(means.index, means.values):
        axes[i].text(x, v + 0.02, f"{v:.2f}", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

## 5. Feature Engineering

### Encoding Strategy
We apply **ordinal encoding** to categorical columns — preserving the natural order:

| Column | Mapping |
|--------|---------|
| `ai_frequency` | Never=0, Sometimes=1, Often=2, Daily=3 |
| `hours_per_week` | 0h=0, <1h=0.5, 1-3h=2, 4-6h=5, >6h=7 |
| `copy_behavior` | Never=0, Rarely=1, Sometimes=2, Often=3, Always=4 |

Likert scale columns (`improved_grades`, `cannot_complete_without_ai`, `critical_thinking`) are already numeric (1–5).

### Target Variable
Binary classification target:
- **1 (High GPA):** GPA ≥ 3.5
- **0 (Low GPA):** GPA < 3.5


In [ ]:
# ── Ordinal Encoding ──────────────────────────────────────────────────────
FREQ_MAP  = {"Never": 0, "Sometimes": 1, "Often": 2, "Daily": 3}
COPY_MAP  = {"Never": 0, "Rarely": 1, "Sometimes": 2, "Often": 3, "Always": 4}
HOURS_MAP = {"0 hours": 0, "less than 1 hours": 0.5, "1-3 hours": 2,
             "4-6 hours": 5, "More than 6 hours": 7}

df["ai_frequency_enc"]  = df["ai_frequency"].map(FREQ_MAP)
df["hours_per_week_enc"]= df["hours_per_week"].map(HOURS_MAP)
df["copy_behavior_enc"] = df["copy_behavior"].map(COPY_MAP)

# Create binary target
df["gpa_high"] = (df["gpa"] >= 3.5).astype(int)

print("Encoded columns added.")
print("\nTarget distribution:")
print(df["gpa_high"].value_counts().rename({0:"Low GPA (<3.5)", 1:"High GPA (≥3.5)"}))
print(f"\nClass balance: {df['gpa_high'].mean()*100:.1f}% High GPA")

In [ ]:
# ── Correlation Heatmap (after encoding) ──────────────────────────────────
numeric_cols = ["ai_frequency_enc","hours_per_week_enc","copy_behavior_enc",
                "improved_grades","cannot_complete_without_ai","critical_thinking","gpa"]
labels       = ["AI Freq","Hours","Copy","Grades","Depend","Critical","GPA"]

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="YlOrRd",
            xticklabels=labels, yticklabels=labels,
            linewidths=0.5, ax=ax, annot_kws={"size": 9})
ax.set_title("Correlation Heatmap (Encoded Features + GPA)", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.show()

print("\nTop correlations with GPA:")
print(corr["gpa"].drop("gpa").sort_values(ascending=False).to_string())

## 6. Logistic Regression Model

### Why Logistic Regression?
- Target is **binary** (High/Low GPA) → classification task
- Provides interpretable **coefficients** showing feature direction & magnitude
- Appropriate for small datasets (57 samples)

### Setup
- **No train/test split** — as specified; model is fit on the full dataset
- `max_iter=1000` to ensure convergence
- `random_state=42` for reproducibility

> ⚠️ Without a test split, accuracy reflects **training performance** (optimistic). For a real-world deployment, use cross-validation or a held-out test set.


In [ ]:
# ── Define features & target ──────────────────────────────────────────────
FEATURES = [
    "ai_frequency_enc",
    "hours_per_week_enc",
    "copy_behavior_enc",
    "improved_grades",
    "cannot_complete_without_ai",
    "critical_thinking",
]
FEATURE_LABELS = [
    "AI Frequency",
    "Hours / Week",
    "Copy Behavior",
    "Improved Grades",
    "AI Dependence",
    "Critical Thinking",
]

X = df[FEATURES]
y = df["gpa_high"]

print(f"X shape: {X.shape}")
print(f"y distribution:\n{y.value_counts().to_string()}")

In [ ]:
# ── Train Logistic Regression ─────────────────────────────────────────────
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X, y)

y_pred      = model.predict(X)
y_pred_prob = model.predict_proba(X)[:, 1]  # probability of High GPA

print("✅ Model trained!")
print(f"   Intercept : {model.intercept_[0]:.4f}")

## 7. Model Evaluation

### Metrics
| Metric | Description |
|--------|-------------|
| **Accuracy** | % of correct predictions overall |
| **Precision** | Of predicted High-GPA, how many actually were? |
| **Recall** | Of actual High-GPA students, how many did we catch? |
| **F1-Score** | Harmonic mean of Precision & Recall |
| **Confusion Matrix** | TN / FP / FN / TP breakdown |

### Feature Coefficients
Positive coefficient → feature **increases** probability of High GPA  
Negative coefficient → feature **decreases** probability of High GPA


In [ ]:
# ── Classification Report ─────────────────────────────────────────────────
acc = accuracy_score(y, y_pred)
print(f"Overall Accuracy: {acc:.4f}  ({acc*100:.1f}%)")
print()
print(classification_report(y, y_pred, target_names=["Low GPA","High GPA"]))

In [ ]:
# ── Confusion Matrix Plot ─────────────────────────────────────────────────
cm = confusion_matrix(y, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: heatmap
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Pred Low","Pred High"],
            yticklabels=["Actual Low","Actual High"],
            annot_kws={"size": 16, "weight": "bold"}, linewidths=1)
axes[0].set_title(f"Confusion Matrix  (Accuracy = {acc:.1%})", fontweight="bold", fontsize=12)

# Right: Feature coefficients bar
coef_series = pd.Series(model.coef_[0], index=FEATURE_LABELS).sort_values(ascending=True)
colors_bar  = ["#F96167" if v < 0 else "#C9F01E" for v in coef_series]
coef_series.plot(kind="barh", ax=axes[1], color=colors_bar, edgecolor="#1A1A2E", linewidth=1)
axes[1].axvline(0, color="#1A1A2E", linewidth=1.5, linestyle="--")
axes[1].set_title("Feature Coefficients
(Impact on High GPA Probability)", fontweight="bold", fontsize=11)
axes[1].set_xlabel("Coefficient Value")
for i, (val, _) in enumerate(zip(coef_series.values, coef_series.index)):
    offset = 0.01 if val >= 0 else -0.01
    ha     = "left"  if val >= 0 else "right"
    axes[1].text(val + offset, i, f"{val:+.3f}", va="center", ha=ha, fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# ── Coefficient Summary Table ─────────────────────────────────────────────
coef_df = pd.DataFrame({
    "Feature"    : FEATURE_LABELS,
    "Coefficient": model.coef_[0],
    "Direction"  : ["⬆ Positive" if v > 0 else "⬇ Negative" for v in model.coef_[0]],
    "Interpretation": [
        "More frequent AI use → higher GPA probability",
        "More study hours → higher GPA probability",
        "Copying behavior → negligible effect",
        "Perceiving grade improvement → higher GPA",
        "AI dependence feeling → slight positive",
        "Critical thinking rating → slight negative",
    ]
}).sort_values("Coefficient", ascending=False).reset_index(drop=True)

print(coef_df.to_string(index=False))

In [ ]:
# ── Predicted Probabilities Sample ───────────────────────────────────────
result_df = df[FEATURES + ["gpa","gpa_high"]].copy()
result_df["pred_class"]   = y_pred
result_df["pred_prob_high"] = y_pred_prob.round(3)
result_df["correct"]      = (result_df["gpa_high"] == result_df["pred_class"])

print(f"Correct predictions: {result_df['correct'].sum()} / {len(result_df)}")
result_df.head(10)

## 8. Conclusion

### Key Findings

| Finding | Detail |
|---------|--------|
| 🏆 **Strongest predictor** | AI Frequency (coef +0.43) — daily users tend toward higher GPA |
| 🕐 **Second strongest** | Hours/Week (coef +0.41) — more study time using AI correlates with higher GPA |
| 📈 **Grade perception matters** | Students who feel AI improved their grades tend to have higher GPA (coef +0.29) |
| 📋 **Copy behavior** | Near-zero coefficient (−0.008) — copying doesn't noticeably help or hurt |
| 🧠 **Critical thinking** | Slight negative coefficient (−0.034) — unexpected, may reflect overconfidence |
| 🎯 **Model accuracy** | 66.7% on full training data (no test split) |

### Limitations
- **Small sample (n=57):** Results may not generalize well
- **No train/test split:** Reported accuracy is training accuracy (optimistic)
- **Ordinal assumptions:** Encoding assumes equal intervals between categories
- **Survey bias:** Self-reported data may not reflect actual behavior

### Recommendations
1. Collect more data (aim for n ≥ 200)
2. Use cross-validation (e.g., 5-fold) for more reliable performance estimates
3. Try tree-based models (Random Forest, XGBoost) that handle non-linear patterns
4. Consider adding demographic features (year of study, major, etc.)


In [ ]:
print("=" * 55)
print("  AI Survey GPA Analysis — Summary")
print("=" * 55)
print(f"  Dataset     : 57 students, 6 features")
print(f"  Target      : GPA ≥ 3.5 (High) vs < 3.5 (Low)")
print(f"  High GPA    : {df['gpa_high'].sum()} students ({df['gpa_high'].mean()*100:.1f}%)")
print(f"  Low GPA     : {(df['gpa_high']==0).sum()} students ({(1-df['gpa_high'].mean())*100:.1f}%)")
print(f"  Model       : Logistic Regression")
print(f"  Accuracy    : {acc*100:.1f}%")
print(f"  Top Feature : AI Frequency (coef +{model.coef_[0][0]:.3f})")
print("=" * 55)